# Semana 1: Jupyter y repaso de Python científico

**Módulo 0** · Facultad de Ciencias, UNAM

## Objetivos de la sesión

1. Usar JupyterLab con fluidez: celdas, magics y exportación.
2. Repasar NumPy y Matplotlib, en contraste con lo que haremos más adelante
   con cómputo simbólico.
3. Adoptar buenas prácticas de reproducibilidad en notebooks.

## Antes de empezar

Esta clase asume que ya instalaste tu entorno siguiendo
[`docs/instalacion.md`](../../docs/instalacion.md) y el checklist de
[`preparacion.md`](../preparacion/preparacion.md). Si algo de eso no
funcionó, dilo ahora — lo resolvemos antes de seguir.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

## Celdas de código vs. markdown

Un notebook combina dos tipos de celda:

- **Markdown** (como esta): texto, ecuaciones en LaTeX, explicación del
  razonamiento.
- **Código**: lo que se ejecuta.

La regla del curso: cada celda de código va acompañada de markdown que
explica el *por qué*, no solo el *qué*. El resultado de un cálculo sin
contexto no sirve como material reproducible — ni para ti en tres meses, ni
para quien revise tu PR.

## Magics útiles

Los *magics* son comandos especiales de Jupyter/IPython, no Python puro
(empiezan con `%` o `%%`):

| Magic | Para qué sirve |
|---|---|
| `%timeit` | Mide el tiempo de una línea, promediando varias corridas |
| `%%timeit` | Igual, pero para toda la celda |
| `%matplotlib inline` | Muestra las figuras de Matplotlib dentro del notebook |
| `%whos` | Lista las variables definidas hasta el momento, con su tipo |

No son parte del lenguaje Python — no funcionan si exportas la celda a un
script `.py` sin quitarlos primero (ver la sección de exportación, más
abajo).

In [ ]:
# Dos formas de construir la misma lista de cuadrados: ¿cuál es más rápida?
%timeit cuadrados = [i**2 for i in range(10_000)]

## TODO en clase 1

Compara, con `%%timeit` (una celda por enfoque, ya que el magic mide toda la
celda), dos formas de sumar los primeros $10^6$ enteros:

1. Con la función incorporada `sum(range(10**6))`.
2. Con un `for` que acumula el resultado en una variable.

¿Cuál es más rápida? ¿Por qué crees que pasa eso?

In [ ]:
# TODO en clase: mide con %%timeit el enfoque con sum()


In [ ]:
# TODO en clase: mide con %%timeit el enfoque con un for acumulador


## Extensiones útiles de JupyterLab

No las vamos a instalar en vivo (la instalación vive en
`docs/instalacion.md`, y en clase no usamos `!pip install`). Solo para que
las conozcas:

- **Table of Contents** — navega el notebook por sus encabezados `##`.
- **Variable Inspector** — muestra las variables activas sin escribir
  `%whos` cada vez.
- **Spellchecker** — revisa ortografía en las celdas de markdown.

## Repaso rápido: NumPy

Un arreglo de NumPy (`np.ndarray`) no es una lista de Python: todos sus
elementos son del mismo tipo, y las operaciones se aplican **vectorizadas**
— elemento a elemento, sin escribir el `for` explícitamente. Eso lo hace
mucho más rápido para cómputo numérico, y es la base de casi todo lo que
vamos a graficar en el curso.

In [ ]:
velocidades = np.linspace(0, 10, 5)  # 5 valores de v, en m/s

def energia_cinetica(v, masa=1.0):
    return 0.5 * masa * v**2

energia_cinetica(velocidades)  # se aplica a los 5 valores a la vez, sin bucle


## TODO en clase 2

Un oscilador armónico simple tiene posición

$$x(t) = A\cos(\omega t)$$

Usando NumPy (nada de símbolos todavía):

1. Define `A = 2.0` y `omega = 3.0`.
2. Genera con `np.linspace` un arreglo `t` de 200 puntos entre `0` y `10`.
3. Calcula `x` aplicando la fórmula de arriba de forma vectorizada.

In [ ]:
# TODO en clase: define A, omega, genera t con np.linspace y calcula x
A = ...
omega = ...
t = ...
x = ...


## Repaso rápido: Matplotlib

La forma más explícita de crear una figura es pidiendo directamente los
objetos `fig` (la figura completa) y `ax` (los ejes donde se dibuja):

```python
fig, ax = plt.subplots()
ax.plot(...)
```

En física computacional, **siempre** etiqueta los ejes con su cantidad y
unidad (`ax.set_xlabel("t [s]")`, no solo `"t"`) — una gráfica sin unidades
no es información, es una curiosidad.

## TODO en clase 3

Grafica el `x(t)` que calculaste arriba: eje horizontal `t`, eje vertical
`x`, con etiquetas de ejes (incluyendo unidades arbitrarias, `"t [u.a.]"` y
`"x [u.a.]"`) y un título.

In [ ]:
# TODO en clase: grafica t vs. x con etiquetas de ejes y título
fig, ax = plt.subplots()


## Contraste con cómputo simbólico (demo motivacional)

Esto es solo una demostración — **no se espera que ustedes escriban código
todavía**. La primera vez que abramos SymPy formalmente será en la
semana 4.

Con NumPy acabamos de calcular *valores numéricos* de $x(t)$ para un $A$ y
un $\omega$ concretos. SymPy, en cambio, puede resolver la ecuación de
movimiento del oscilador de forma simbólica — sin fijar ningún valor —
y darnos la fórmula general:

In [ ]:
import sympy as sp

sp.init_printing()

t_sym = sp.symbols('t', real=True)
omega_sym = sp.symbols('omega', positive=True)
x_sym = sp.Function('x')

ecuacion = sp.Eq(x_sym(t_sym).diff(t_sym, 2) + omega_sym**2 * x_sym(t_sym), 0)
sp.dsolve(ecuacion, x_sym(t_sym))

Nota la diferencia: arriba, `x` es un arreglo de 200 números concretos.
Aquí, la salida es una **fórmula** válida para cualquier condición inicial
— ese es el salto que vamos a dar a partir de la semana 4.

## Buenas prácticas: reproducibilidad y orden de ejecución

Los notebooks permiten ejecutar celdas en cualquier orden — y eso es
también su mayor riesgo. Un notebook que "funciona" en tu sesión actual
puede fallar para cualquier otra persona si el orden real de ejecución no
coincide con el orden en que aparecen las celdas.

| Síntoma | Causa típica |
|---|---|
| `NameError` al reabrir el notebook | Una variable se definió en una celda que luego borraste o moviste |
| El resultado cambia según quién lo corre | Una celda de arriba se ejecutó dos veces (p. ej. un contador que se incrementa) |
| Funciona en tu máquina, falla en la revisión | Nunca se probó desde un kernel limpio |

Por eso, antes de dar un notebook por terminado (y **siempre** antes de
abrir un PR): `Kernel → Restart & Run All`. Si eso no corre limpio de
principio a fin, el notebook no está listo.

## Exportación: `nbconvert`

A veces conviene convertir un notebook a otro formato: un script `.py` para
correrlo fuera de Jupyter, o HTML para compartirlo con alguien que no tiene
Jupyter instalado. La herramienta es `nbconvert`, desde la terminal (no
desde una celda del notebook — en este curso no usamos `!` para comandos de
shell dentro de un notebook):

```bash
jupyter nbconvert --to script mi_notebook.ipynb   # -> mi_notebook.py
jupyter nbconvert --to html mi_notebook.ipynb     # -> mi_notebook.html
```

Es también lo que usamos para *verificar* que un notebook ejecuta limpio
antes de un PR:

```bash
jupyter nbconvert --to notebook --execute --inplace mi_notebook.ipynb
```

## Resumen

Hoy repasamos JupyterLab (celdas, magics, extensiones), NumPy y Matplotlib
como base numérica, y por qué la reproducibilidad importa desde la primera
clase. También vimos, solo como demostración, un adelanto de lo que SymPy
puede hacer a partir de la semana 4.

**Tarea de esta semana:** [`tarea-01.ipynb`](../tarea/tarea-01.ipynb) —
entrega antes de la clase de la Semana 2.

**Próxima clase — Semana 2:** Programación orientada a objetos en Python.